# Phase 3 – Statistical Analysis

OLS regression, GWR, and sensitivity analysis for California schools.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.statistical_analysis import run_phase3, load_caaspp, aggregate_caaspp, build_ols_model
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from config import DATA_PROCESSED

## Run Full Phase 3 Pipeline

In [ ]:
results = run_phase3()
print(results)

## Load California Data for Exploration

In [ ]:
for p in [DATA_PROCESSED / 'schools_with_demographics.gpkg',
          DATA_PROCESSED / 'schools_noise_classified.gpkg']:
    if p.exists():
        schools = gpd.read_file(p)
        break
state_col = next((c for c in schools.columns if c in ('STABR','ST','STATE_ABBR','STABBR')), None)
ca = schools[schools[state_col]=='CA'].copy() if state_col else schools.copy()
print(f'California schools: {len(ca):,}')
ca[['noise_db','noise_tier','dist_highway_m']].describe()

## Noise vs. Academic Performance

*(Requires CAASPP data download and merge)*

In [ ]:
if 'pct_proficient_mean' in ca.columns and ca['pct_proficient_mean'].notna().sum() > 30:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    valid = ca[['noise_db','pct_proficient_mean']].dropna()
    axes[0].scatter(valid['noise_db'], valid['pct_proficient_mean'], alpha=0.3, s=8, c='#3498db')
    m, b = np.polyfit(valid['noise_db'], valid['pct_proficient_mean'], 1)
    xs = np.linspace(valid['noise_db'].min(), valid['noise_db'].max(), 100)
    axes[0].plot(xs, m*xs+b, color='red', lw=2, label=f'slope={m:.2f}')
    axes[0].axvline(55, color='orange', ls='--', label='WHO threshold')
    axes[0].set_xlabel('Noise Level (dB)')
    axes[0].set_ylabel('% Proficient')
    axes[0].set_title('Noise vs. Academic Proficiency')
    axes[0].legend()

    COLORS = ['#2ecc71','#f1c40f','#e67e22','#e74c3c']
    data_by_tier = [ca[ca['noise_tier']==i]['pct_proficient_mean'].dropna().values for i in range(1,5)]
    bp = axes[1].boxplot(data_by_tier, labels=['T1','T2','T3','T4'], patch_artist=True, showfliers=False)
    for patch, color in zip(bp['boxes'], COLORS):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    axes[1].set_title('Proficiency by Noise Tier')
    axes[1].set_ylabel('% Meeting/Exceeding Standard')

    plt.tight_layout()
    plt.show()
else:
    print('Download CAASPP data and run Phase 2 merge to enable this analysis.')

## OLS Regression Summary

In [ ]:
ols_path = DATA_PROCESSED / 'ols_results.txt'
if ols_path.exists():
    print(open(ols_path).read())
else:
    print('Run Phase 3 with CAASPP data to generate OLS output.')

## GWR Local Coefficient Map

In [ ]:
gwr_path = DATA_PROCESSED / 'gwr_coefficients.csv'
if gwr_path.exists():
    coefs = pd.read_csv(gwr_path)
    print(coefs.describe())
    coefs['noise_db'].hist(bins=30, color='#e74c3c', edgecolor='white')
    plt.axvline(0, color='black', lw=1.5, ls='--')
    plt.title('GWR: Distribution of Local Noise Coefficients')
    plt.xlabel('Coefficient Value (effect of noise on proficiency)')
    plt.show()
else:
    print('Install mgwr (pip install mgwr) and run Phase 3 to generate GWR output.')